In [ ]:
import os

import numpy as np
import pandas as pd
from scipy.stats import sem


def mae_and_sem(preds: np.ndarray, trues: np.ndarray):
    abs_error = np.abs(preds - trues)
    mae = np.mean(abs_error)
    mae_sem = sem(abs_error)
    return mae, mae_sem


def load_outputs(logdir: str):
    return np.load(os.path.join(logdir, "test_outputs.npy"))

In [ ]:
test_index_path = "../data/index/test.parquet"
sbp_logdir = "../calib_free/outputs/calib_free/inception-scratch-sbp"
dbp_logdir = "../calib_free/outputs/calib_free/inception-scratch-dbp"

In [ ]:
df = pd.read_parquet(test_index_path)
gt_sbps = df["sbp"].values
gt_dbps = df["dbp"].values
is_unstable = df["is_unstable"].values

pred_sbps = load_outputs(sbp_logdir)
pred_dbps = load_outputs(dbp_logdir)

# 1) Whole
print("Standard evaluation (MAE)")
sbp_mae, sbp_mae_sem = mae_and_sem(pred_sbps, gt_sbps)
dbp_mae, dbp_mae_sem = mae_and_sem(pred_dbps, gt_dbps)
print(f"\tSBP: {sbp_mae:.2f} ± {1.96 * sbp_mae_sem:.2f}")
print(f"\tDBP: {dbp_mae:.2f} ± {1.96 * dbp_mae_sem:.2f}")

# 2) Unstable BP intervals
print("Fluctuation-aware evaluation (MAE)")
sbp_mae_unstable, sbp_mae_sem_unstable = mae_and_sem(pred_sbps[is_unstable], gt_sbps[is_unstable])
dbp_mae_unstable, dbp_mae_sem_unstable = mae_and_sem(pred_dbps[is_unstable], gt_dbps[is_unstable])
print(f"\tSBP: {sbp_mae_unstable:.2f} ± {1.96 * sbp_mae_sem_unstable:.2f}")
print(f"\tDBP: {dbp_mae_unstable:.2f} ± {1.96 * dbp_mae_sem_unstable:.2f}")